In [1]:
import sys; sys.path.insert(0, "/home/sagemaker-user/aep")
from prep.common import *

g = read_shp("Customer_Clip")
print(g.crs, len(g))
g.head(3).T

EPSG:4326 117642


,0,1,2
id,132055394,132055398,132055401
critical_c,N/A,N/A,N/A
special_op,None,None,None
aep_locati,37800295D00119,37800295D00045,37800295D00045
aep_xfmr_l,37800295D00119,37800295D00045,37800295D00045
road_id,16698237,16034031,16034031
aep_string,1,1,1
customer_c,1,1,1
address,None,None,None
road_name,GARDNER,CANNON,CANNON


In [2]:
cust = clean_nulls(add_latlon(g))
premise = pd.DataFrame({
    "premise_id": as_id(cust["premise_nu"]),
    "lat": cust["lat"], "lon": cust["lon"],
    "circuit_id": cust["circuit_id"].map(to_circuit),
    "transformer_loc": cust["aep_xfmr_l"],
    "service_type": cust["kind_of_se"],
    "city": cust["city"].str.title(),
})
# INTERNAL ONLY: Life Support count per transformer (never premise-level, never uploaded)
ls_per_xf = (cust.assign(ls=cust["critical_c"].eq("LS Life Support").fillna(False))
                 .groupby("aep_xfmr_l")["ls"].sum().rename("life_support_count").reset_index())

check(premise, "premise", key="premise_id")
print(premise.service_type.value_counts(dropna=False))
print("life support total:", int(ls_per_xf.life_support_count.sum()))
premise.head()

--- premise: 117,642 rows
  duplicate keys: 0
  outside study area: 0
  PRIVACY OK
  null %: {'service_type': 0.3}
service_type
Overhead       64824
Underground    52419
<NA>             399
Name: count, dtype: Int64
life support total: 113


,premise_id,lat,lon,circuit_id,transformer_loc,service_type,city
0,247057282,37.234343,-80.212643,CIR27,37800295D00119,Underground,Elliston
1,885826470,37.236093,-80.212327,CIR27,37800295D00045,Overhead,Elliston
2,365173931,37.236099,-80.212364,CIR27,37800295D00045,Overhead,Elliston
3,073240485,37.236171,-80.212215,CIR27,37800295D00045,Overhead,Elliston
4,944922967,37.296464,-79.762315,CIR5,37800234D00042,Underground,Pound


In [3]:
rows = []
with open(f"{RAW}/matched_upstream.txt", encoding="latin1") as f:
    for line in f:
        parts = line.rstrip("\r\n").split("\t")
        for step, item in enumerate(p for p in parts[1:] if p):
            kind, _gid, desc, loc = item.split("*")[:4]
            breaker = kind == "eo_circuit"
            rows.append((parts[0], step, None if breaker else loc,
                         {"Distribution": "Transformer", "Step": "Step Transformer"}.get(desc, desc),
                         to_circuit(loc) if breaker else None))
chain = pd.DataFrame(rows, columns=["premise_id", "step", "device_loc", "device_type", "circuit_id"])
premise["has_chain"] = premise.premise_id.isin(chain.premise_id)

print(f"{len(chain):,} chain rows; premises without a chain: {(~premise.has_chain).sum()}")
print(chain.device_type.value_counts())
chain.head(8)

490,090 chain rows; premises without a chain: 47
device_type
Fuse                     154821
Circuit Breaker          117599
Transformer              117599
Recloser (Hydraulic)      64910
Recloser (Electronic)     28399
Step Transformer           6614
Sectionalizer               148
Name: count, dtype: int64


,premise_id,step,device_loc,device_type,circuit_id
0,551102222,0,37800368000008,Transformer,None
1,551102222,1,37800343D00129,Recloser (Hydraulic),None
2,551102222,2,37800319D40521,Recloser (Hydraulic),None
3,551102222,3,37800319B30077,Recloser (Hydraulic),None
4,551102222,4,37800296B46096,Recloser (Electronic),None
5,551102222,5,None,Circuit Breaker,CIR27
6,602845057,0,37800368000011,Transformer,None
7,602845057,1,37800343D00129,Recloser (Hydraulic),None


In [4]:
xf = clean_nulls(add_latlon(read_shp("Transformer_Clip")))
kva_text = xf[["a_phase_sp", "b_phase_sp", "c_phase_sp"]].bfill(axis=1).iloc[:, 0]
transformer = pd.DataFrame({
    "transformer_id": as_id(xf["id"]), "transformer_loc": xf["location_n"],
    "circuit_id": xf["aep_circui"].map(to_circuit), "lat": xf["lat"], "lon": xf["lon"],
    "kva": pd.to_numeric(kva_text.str.extract(r"([\d.]+)\s*kVA")[0], errors="coerce"),
    "mounting": xf["mounting"], "phase": xf["bank_confi"],
    "upstream_loc": xf["upstream_l"], "customers": pd.to_numeric(xf["aep_custom"], errors="coerce"),
})
check(transformer, "transformer", key="transformer_loc")
print(transformer.kva.describe().round(1))
print(transformer.mounting.value_counts(dropna=False))
transformer.head()

--- transformer: 26,959 rows
  duplicate keys: 9
  outside study area: 0
  PRIVACY OK
  null %: {'circuit_id': 0.2, 'phase': 12.7, 'upstream_loc': 2.0}
count    26949.0
mean        54.7
std        127.6
min          0.5
25%         25.0
50%         25.0
75%         50.0
max       2500.0
Name: kva, dtype: Float64
mounting
Overhead       20303
Underground     6656
Name: count, dtype: Int64


,transformer_id,transformer_loc,circuit_id,lat,lon,kva,mounting,phase,upstream_loc,customers
0,144901302,37800045000079,CIR60,37.479176,-79.665325,25.0,Overhead,Single Phase,37800045000064,2
1,144901305,37800045000065,CIR60,37.476376,-79.659069,25.0,Overhead,Single Phase,37800045000064,1
2,144901308,37800045000063,CIR60,37.473508,-79.660848,25.0,Overhead,Single Phase,37800045000012,1
3,147843303,37800045A20092,CIR60,37.473333,-79.658805,25.0,Underground,Single Phase,37800045000064,1
4,144901311,37800045000031,CIR60,37.471848,-79.662029,10.0,Overhead,Single Phase,37800045000012,2


In [5]:
# one row per transformer location (keep the largest unit of a bank)
transformer = (transformer.sort_values("kva", ascending=False, na_position="last")
                          .drop_duplicates("transformer_loc").reset_index(drop=True))
print("transformers after removing duplicates:", len(transformer))

parts = []
for name in ["Fuse_Bank_Clip", "Recloser_Bank_Clip", "Switch_Clip"]:
    g = clean_nulls(add_latlon(read_shp(name)))
    rating = g[[c for c in ["a_phase_sp", "b_phase_sp", "c_phase_sp"] if c in g]].bfill(axis=1).iloc[:, 0]
    parts.append(pd.DataFrame({
        "device_id": as_id(g["id"]), "device_loc": g["location_n"], "device_type": g["device_typ"],
        "circuit_id": g["aep_circui"].map(to_circuit), "lat": g["lat"], "lon": g["lon"],
        "normal_state": g["normal_sta"], "rating": rating,
        "upstream_loc": g["upstream_l"] if "upstream_l" in g else pd.NA,
        "customers_downstream": pd.to_numeric(g["aep_custom"], errors="coerce"),
        "automated": (~g["esri_defau"].fillna("").str.contains("No Automation")) if name != "Fuse_Bank_Clip" else False,
    }))
device = pd.concat(parts, ignore_index=True)
check(device, "device", key="device_loc")
print(device.device_type.value_counts())

# do the chain devices exist in our device/transformer tables?
known = set(device.device_loc) | set(transformer.transformer_loc)
dl = chain.device_loc.dropna()
print("chain devices found:", round(dl.isin(known).mean() * 100, 1), "%")
device.head()

transformers after removing duplicates: 26950
--- device: 8,847 rows
  duplicate keys: 1057
  outside study area: 0
  PRIVACY OK
  null %: {'rating': 0.1, 'upstream_loc': 10.4}
device_type
Fuse                     7616
Switch                    900
Recloser (Hydraulic)      289
Recloser (Electronic)      42
Name: count, dtype: Int64
chain devices found: 97.5 %


,device_id,device_loc,device_type,circuit_id,lat,lon,normal_state,rating,upstream_loc,customers_downstream,automated
0,139702311,37800045000064,Fuse,CIR60,37.475536,-79.659419,Closed,FUSE 10 T LB,37800045000063,4,False
1,140156416,37800045000064,Fuse,CIR60,37.475553,-79.659423,Closed,FUSE 10 T LB,37800045000063,4,False
2,138459045,37800045000063,Fuse,CIR60,37.473540,-79.660870,Closed,FUSE 15 T LB,37800045000012,4,False
3,139344975,37800044000111,Fuse,CIR60,37.467239,-79.666592,Closed,FUSE 10 T LB,37800045000012,1,False
4,138459048,37800045000083,Fuse,CIR60,37.468900,-79.661185,Closed,FUSE 10 T LB,37800045000012,2,False


In [6]:
# one row per device location; keep how many units (phases) the bank has
units = device.groupby("device_loc").size().rename("units")
device = device.drop_duplicates("device_loc").merge(units, on="device_loc").reset_index(drop=True)
print("devices after merging banks:", len(device))

# each customer's zone = their first fuse / recloser / sectionalizer going upstream
PROTECTIVE = ["Fuse", "Recloser (Hydraulic)", "Recloser (Electronic)", "Sectionalizer"]
first = (chain[chain.device_type.isin(PROTECTIVE)].sort_values("step")
              .groupby("premise_id").first().reset_index()[["premise_id", "device_loc"]]
              .rename(columns={"device_loc": "zone_id"}))
premise = premise.drop(columns="zone_id", errors="ignore").merge(first, on="premise_id", how="left")
premise["zone_id"] = premise["zone_id"].fillna("BREAKER_" + premise["circuit_id"].fillna("UNKNOWN"))

zone = (premise.groupby("zone_id").agg(customers=("premise_id", "size"), circuit_id=("circuit_id", "first"))
               .reset_index()
               .merge(device[["device_loc", "device_type", "lat", "lon"]],
                      left_on="zone_id", right_on="device_loc", how="left")
               .drop(columns="device_loc"))

# each transformer's zone = the most common zone of its customers
xf_zone = (premise.dropna(subset=["transformer_loc"]).groupby("transformer_loc")["zone_id"]
                  .agg(lambda s: s.mode().iat[0]).rename("zone_id").reset_index())
transformer = transformer.drop(columns="zone_id", errors="ignore").merge(xf_zone, on="transformer_loc", how="left")

print(f"{len(zone):,} zones | median customers per zone: {zone.customers.median():.0f} | largest: {zone.customers.max():,}")
print("customers fed straight from a breaker (no fuse/recloser):", premise.zone_id.str.startswith("BREAKER_").sum())
print("transformers with a zone:", round(transformer.zone_id.notna().mean() * 100, 1), "%")
zone.sort_values("customers", ascending=False).head()

devices after merging banks: 7790
6,664 zones | median customers per zone: 6 | largest: 354
customers fed straight from a breaker (no fuse/recloser): 7041
transformers with a zone: 93.9 %


,zone_id,customers,circuit_id,device_type,lat,lon
2073,37800229C11216,354,CIR95,Fuse,37.307444,-79.976983
2583,37800250D42037,347,CIR31,Fuse,37.271646,-80.087101
2309,37800231B47788,345,CIR64,Fuse,37.294151,-79.903780
4401,37800300D37502,339,CIR35,Fuse,37.239609,-80.007415
5648,37800348A41761,324,CIR18,Fuse,37.198235,-80.029588


In [7]:
for df, n in [(premise, "premise"), (chain, "chain"), (transformer, "transformer"),
              (device, "device"), (zone, "zone")]:
    save(df, n, "batch1")
save(ls_per_xf, "life_support_per_transformer", "batch1", local_only=True)   # stays on this machine

!aws s3 sync ~/aep/processed/batch1 s3://sagemaker-aep-outage-indiansassemble/processed/batch1

saved premise: 117,642 rows -> /home/sagemaker-user/aep/processed/batch1
saved chain: 490,090 rows -> /home/sagemaker-user/aep/processed/batch1
saved transformer: 26,950 rows -> /home/sagemaker-user/aep/processed/batch1
saved device: 7,790 rows -> /home/sagemaker-user/aep/processed/batch1
saved zone: 6,664 rows -> /home/sagemaker-user/aep/processed/batch1
saved life_support_per_transformer: 25,309 rows -> /home/sagemaker-user/aep/local_only/batch1
upload: processed/batch1/chain.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch1/chain.parquet
upload: processed/batch1/device.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch1/device.parquet
upload: processed/batch1/zone.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch1/zone.parquet
upload: processed/batch1/transformer.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch1/transformer.parquet
upload: processed/batch1/premise.parquet to s3://sagemaker-aep-outage-indian

In [8]:
del g, cust, xf, parts, rows          # free memory from raw Batch 1 inputs
import gc; gc.collect()

dmg = clean_nulls(read_shp("arcos_prod_damage_09_28_Clip"))
dmg = dmg.dropna(axis=1, how="all")   # drops the completely blank columns (Event, District, Status, crew...)
rt = pd.to_datetime(dmg["Report_Tim"], errors="coerce").dt.normalize()
q = pd.to_numeric(dmg["Media"], errors="coerce").fillna(0).astype(int)
repaired = dmg["Repaired"].eq("Yes").fillna(False)

damage = pd.DataFrame({
    "damage_id": range(len(dmg)),
    "event": np.select([rt.between("2025-02-10", "2025-02-20"), rt.between("2024-09-25", "2024-10-05")],
                       ["feb2025", "sep2024"], "other"),
    "report_date": rt.dt.date,
    "repaired": repaired,
    "repaired_date": pd.to_datetime(dmg["Repaired_D"], errors="coerce").dt.date.where(repaired),
    "circuit_id": dmg["Circuit"].map(to_circuit),
    "circuit_raw": dmg["Circuit"],
    "substation": dmg["Substation"],
    "asset_type": dmg["Asset_Type"],
    "damage_type": dmg["Inform_Wor"],
    "quantity": q.clip(lower=1), "quantity_was_zero": q.eq(0),
    "access_issue": dmg["Access_Iss"].map({"Yes": True, "No": False}),
    "special_equipment": dmg["Special_Eq"],
    "lat": pd.to_numeric(dmg["Latitude"], errors="coerce").round(6),
    "lon": pd.to_numeric(dmg["Longitude"], errors="coerce").round(6),
})
check(damage, "damage", key="damage_id")
print(damage.event.value_counts())
print("non-standard circuit names:", damage.circuit_id.isna().sum(), damage.loc[damage.circuit_id.isna(), "circuit_raw"].unique())
print(damage.asset_type.value_counts(dropna=False))
print("access issue:", damage.access_issue.value_counts(dropna=False).to_dict())
damage.head()

--- damage: 1,659 rows
  duplicate keys: 0
  outside study area: 0
  PRIVACY OK
  null %: {'repaired_date': 4.6, 'circuit_id': 0.8, 'asset_type': 0.1, 'access_issue': 0.1, 'special_equipment': 82.8}
event
feb2025    1254
other       211
sep2024     194
Name: count, dtype: int64
non-standard circuit names: 14 <StringArray>
['TINKER', 'PENN FOREST', 'SERVICE CENTER']
Length: 3, dtype: string
asset_type
primary             704
Pole                355
Fuse Bank           280
secondary           156
Transformer          74
Recloser Bank        58
Switch               19
Capacitor Bank        6
Light                 5
<NA>                  1
Step Transformer      1
Name: count, dtype: Int64
access issue: {False: 1123, True: 535, nan: 1}


,damage_id,event,report_date,repaired,repaired_date,circuit_id,circuit_raw,substation,asset_type,damage_type,quantity,quantity_was_zero,access_issue,special_equipment,lat,lon
0,0,other,2025-05-31,False,NaN,CIR60,CIR60,STATION2,primary,Damaged Wire,2,False,NaN,<NA>,37.347863,-79.767591
1,1,other,2025-03-16,False,NaN,CIR65,CIR65,STATION28,Pole,Damaged Pole,1,False,False,<NA>,37.244359,-79.991150
2,2,other,2025-03-16,True,2025-03-17,CIR100,CIR100,STATION28,Pole,Damaged Pole,1,False,False,<NA>,37.243450,-79.981513
3,3,other,2025-03-16,True,2025-03-17,CIR100,CIR100,STATION28,Pole,Damaged Pole,3,False,False,<NA>,37.243913,-79.981443
4,4,other,2025-03-16,True,2025-03-17,CIR100,CIR100,STATION28,primary,Damaged Wire,1,False,False,<NA>,37.243696,-79.981498


In [9]:
t = points(transformer[["transformer_loc", "circuit_id", "zone_id", "lat", "lon"]]
           .rename(columns={"circuit_id": "xf_circuit", "zone_id": "xf_zone"}))
d = points(damage)
m = gpd.sjoin_nearest(d, t.drop(columns=["lat", "lon"]), how="left", max_distance=300, distance_col="match_dist_m")
m = m[~m.index.duplicated()]                         # if two transformers tie, keep one

print("matched within 300 m:", round(m.transformer_loc.notna().mean() * 100, 1), "%")
print("match distance (m):", m.match_dist_m.describe().round(1).to_dict())

# fix the 14 non-standard circuit names using the matched transformer's circuit
m["circuit_id"] = m["circuit_id"].fillna(m["xf_circuit"])
print("circuits still missing:", m.circuit_id.isna().sum())

LOCAL = ["Transformer", "Step Transformer", "secondary"]   # affects one transformer only
IGNORE = ["Light", "Capacitor Bank"]                      # no customer outage
m["outage_loc"] = np.where(m.asset_type.isin(LOCAL), m.transformer_loc, m.xf_zone)
m.loc[m.asset_type.isin(IGNORE), "outage_loc"] = None

damage = pd.DataFrame(m.drop(columns=["geometry", "index_right"], errors="ignore"))
print("reports with an outage area:", damage.outage_loc.notna().sum(), "of", len(damage))
print(damage.groupby("event").outage_loc.apply(lambda s: s.notna().mean().round(3)))
damage[["event", "asset_type", "transformer_loc", "xf_zone", "outage_loc", "match_dist_m"]].head()

matched within 300 m: 99.2 %
match distance (m): {'count': 1646.0, 'mean': 57.1, 'std': 48.9, 'min': 0.0, '25%': 26.6, '50%': 46.3, '75%': 75.4, 'max': 297.0}
circuits still missing: 0
reports with an outage area: 1533 of 1659
event
feb2025    0.915
other      0.938
sep2024    0.964
Name: outage_loc, dtype: float64


,event,asset_type,transformer_loc,xf_zone,outage_loc,match_dist_m
0,other,primary,37800186000028,37800186000023,37800186000023,209.080385
1,other,Pole,37800301A20504,37800301A20505,37800301A20505,44.268654
2,other,Pole,37800301A41615,37800301A41587,37800301A41587,51.743258
3,other,Pole,37800301A41615,37800301A41587,37800301A41587,0.000000
4,other,primary,37800301A41615,37800301A41587,37800301A41587,24.565223


In [10]:
# 1) OUTAGE LABELS: a customer was out in an event if any equipment on their chain (or their zone) was an outage area
hits = damage.dropna(subset=["outage_loc"]).groupby("event")["outage_loc"].apply(set)
chain_sets = chain.dropna(subset=["device_loc"]).groupby("premise_id")["device_loc"].apply(set)
chain_sets = chain_sets.reindex(premise.premise_id)
sets = [(s if isinstance(s, set) else set()) | {z} for s, z in zip(chain_sets.values, premise.zone_id.values)]

outage_labels = pd.DataFrame({"premise_id": premise.premise_id.values, "has_chain": premise.has_chain.values})
for ev in ["feb2025", "sep2024", "other"]:
    s = hits.get(ev, set())
    outage_labels[f"out_{ev}"] = [bool(x & s) for x in sets]
print("share of customers out per event:")
print(outage_labels.filter(like="out_").mean().round(3))

# 2) DAMAGE COUNTS per zone and event, by damage category (for the expected-damage model)
CAT = {"Pole": "pole", "primary": "primary_wire", "Transformer": "transformer", "Step Transformer": "transformer",
       "Fuse Bank": "device", "Recloser Bank": "device", "Switch": "device", "secondary": "secondary"}
damage["damage_cat"] = damage["asset_type"].map(CAT)
zone_damage_counts = (damage.dropna(subset=["damage_cat", "xf_zone"])
                      .pivot_table(index=["xf_zone", "event"], columns="damage_cat", values="quantity",
                                   aggfunc="sum", fill_value=0)
                      .reset_index().rename(columns={"xf_zone": "zone_id"}))
zone_damage_counts.columns.name = None
print(f"\nzone_damage_counts: {len(zone_damage_counts):,} zone-event rows")
print(zone_damage_counts.drop(columns=["zone_id", "event"]).sum())

# 3) ACCESS LABELS: one row per damage report with a known yes/no (for the access model)
access_labels = damage.loc[damage.access_issue.notna(),
                           ["damage_id", "event", "access_issue", "asset_type", "xf_zone", "circuit_id", "lat", "lon"]]
print(f"\naccess_labels: {len(access_labels):,} rows, share with access issue: {access_labels.access_issue.mean():.1%}")
zone_damage_counts.head()

share of customers out per event:
out_feb2025    0.189
out_sep2024    0.037
out_other      0.065
dtype: float64

zone_damage_counts: 719 zone-event rows
device          358
pole            484
primary_wire    851
secondary       234
transformer      88
dtype: int64

access_labels: 1,658 rows, share with access issue: 32.3%


,zone_id,event,device,pole,primary_wire,secondary,transformer
0,37800045000012,feb2025,1,0,0,0,0
1,37800068000014,sep2024,0,0,2,0,0
2,37800087D00014,other,0,0,0,1,0
3,37800091000010,sep2024,0,3,0,0,0
4,37800091000022,feb2025,1,0,0,0,0


In [11]:
for df, n in [(damage, "damage"), (outage_labels, "outage_labels"),
              (zone_damage_counts, "zone_damage_counts"), (access_labels, "access_labels")]:
    save(df, n, "batch2")
!aws s3 sync ~/aep/processed/batch2 s3://sagemaker-aep-outage-indiansassemble/processed/batch2

saved damage: 1,659 rows -> /home/sagemaker-user/aep/processed/batch2
saved outage_labels: 117,642 rows -> /home/sagemaker-user/aep/processed/batch2
saved zone_damage_counts: 719 rows -> /home/sagemaker-user/aep/processed/batch2
saved access_labels: 1,658 rows -> /home/sagemaker-user/aep/processed/batch2
upload: processed/batch2/access_labels.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch2/access_labels.parquet
upload: processed/batch2/zone_damage_counts.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch2/zone_damage_counts.parquet
upload: processed/batch2/damage.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch2/damage.parquet
upload: processed/batch2/outage_labels.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch2/outage_labels.parquet


In [1]:
import sys; sys.path.insert(0, "/home/sagemaker-user/aep")
from prep.common import *

transformer = pd.read_parquet(f"{OUT}/batch1/transformer.parquet")
t = points(transformer[["transformer_loc", "zone_id", "lat", "lon"]].rename(columns={"zone_id": "xf_zone"}))

p = clean_nulls(add_latlon(read_shp("Pole_Clip")))
year = pd.to_datetime(p["date_insta"], errors="coerce").dt.year
WOOD = ["Pine", "Wood", "Cedar", "Ponderosa Pine", "Douglas Fir"]
pole = pd.DataFrame({
    "pole_id": as_id(p["id"]), "lat": p["lat"], "lon": p["lon"],
    "install_year": year, "age_years": 2026 - year,
    "material_group": p["material"].where(~p["material"].isin(WOOD), "wood").str.lower(),
    "height_ft": pd.to_numeric(p["height"].str.extract(r"(\d+)")[0], errors="coerce"),
    "pole_class": p["class"],
})
pz = gpd.sjoin_nearest(points(pole), t[["xf_zone", "geometry"]], how="left", max_distance=500)
pole["zone_id"] = pz[~pz.index.duplicated()]["xf_zone"].values
del p, pz; import gc; gc.collect()

check(pole, "pole", key="pole_id")
print(pole.material_group.value_counts(dropna=False).head(8))
print("install year known:", round(pole.install_year.notna().mean() * 100, 1), "%  | poles with a zone:",
      round(pole.zone_id.notna().mean() * 100, 1), "%")
pole.head()

--- pole: 107,013 rows
  duplicate keys: 0
  outside study area: 0
  PRIVACY OK
  null %: {'install_year': 49.8, 'age_years': 49.8, 'height_ft': 17.9, 'pole_class': 22.2, 'zone_id': 29.4}
material_group
wood          102024
fiberglass      3543
aluminium        644
steel            520
ornamental       120
concrete         114
<NA>              48
Name: count, dtype: Int64
install year known: 50.2 %  | poles with a zone: 70.6 %


,pole_id,lat,lon,install_year,age_years,material_group,height_ft,pole_class,zone_id
0,132454303,37.453631,-79.603829,1956.0,70.0,wood,35,5,NaN
1,132454287,37.449783,-79.605203,1956.0,70.0,wood,35,5,NaN
2,132454278,37.449350,-79.605787,1956.0,70.0,wood,35,5,NaN
3,132454092,37.454272,-79.629591,1982.0,44.0,wood,35,5,NaN
4,132454090,37.454103,-79.630341,1982.0,44.0,wood,35,5,NaN


In [2]:
tz = t.dropna(subset=["xf_zone"])[["xf_zone", "geometry"]]
pz = gpd.sjoin_nearest(points(pole.drop(columns="zone_id")), tz, how="left",
                       max_distance=1000, distance_col="zone_dist_m")
pz = pz[~pz.index.duplicated()]
pole["zone_id"] = pz["xf_zone"].values
pole["zone_dist_m"] = pz["zone_dist_m"].round(1).values
print("poles with a zone:", round(pole.zone_id.notna().mean() * 100, 1), "%")
print("distance to zone (m):", pole.zone_dist_m.describe().round(0).to_dict())

poles with a zone: 77.5 %
distance to zone (m): {'count': 82935.0, 'mean': 71.0, 'std': 129.0, 'min': 0.0, '25%': 15.0, '50%': 44.0, '75%': 71.0, 'max': 999.0}


In [3]:
lines = []
for name, level in [("Primary_ExportFeatures", "primary"), ("Secondary_ExportFeatures", "secondary")]:
    g = gpd.read_file(f"{RAW}/{name}.shp", columns=["aep_circui", "mounting", "nominal_vo"]).to_crs(UTM)
    g = clean_nulls(g)
    lines.append(gpd.GeoDataFrame({
        "level": level,
        "circuit_id": g["aep_circui"].map(to_circuit),
        "mounting": g["mounting"],
        "voltage": g["nominal_vo"],
        "length_m": g.geometry.length.round(1),
    }, geometry=g.geometry.centroid, crs=UTM))
    print(f"{level}: {len(g):,} segments")
    del g; gc.collect()

line = pd.concat(lines, ignore_index=True); del lines
lz = gpd.sjoin_nearest(line, tz, how="left", max_distance=1000)
line["zone_id"] = lz[~lz.index.duplicated()]["xf_zone"].values
ll = line.geometry.to_crs(4326)
line["lat"], line["lon"] = ll.y.round(6), ll.x.round(6)
line = pd.DataFrame(line.drop(columns="geometry")); del lz, ll; gc.collect()

check(line, "line")
print(line.groupby(["level", "mounting"], dropna=False).length_m.sum().div(1000).round(0).rename("km"))
print("lines with a zone:", round(line.zone_id.notna().mean() * 100, 1), "%")
line.head()

primary: 62,788 segments
secondary: 127,403 segments
--- line: 190,191 rows
  outside study area: 0
  PRIVACY OK
  null %: {'circuit_id': 0.4, 'mounting': 0.1, 'voltage': 0.2}
level      mounting   
primary    Overhead       3056.0
           Underground    1015.0
secondary  Overhead       2252.0
           Underground    1309.0
           <NA>              2.0
Name: km, dtype: float64
lines with a zone: 100.0 %


,level,circuit_id,mounting,voltage,length_m,zone_id,lat,lon
0,primary,CIR60,Overhead,12.47 kV,49.3,37800045000064,37.478584,-79.661427
1,primary,CIR60,Overhead,12.47 kV,93.7,37800045000064,37.476732,-79.658773
2,primary,CIR60,Overhead,12.47 kV,81.2,37800045000064,37.477918,-79.660454
3,primary,CIR60,Overhead,12.47 kV,86.3,37800045000064,37.478522,-79.660872
4,primary,CIR60,Overhead,12.47 kV,55.6,37800045000064,37.478212,-79.661793


In [4]:
save(pole, "pole", "batch3")
save(line, "line", "batch3")
!aws s3 sync ~/aep/processed/batch3 s3://sagemaker-aep-outage-indiansassemble/processed/batch3

saved pole: 107,013 rows -> /home/sagemaker-user/aep/processed/batch3
saved line: 190,191 rows -> /home/sagemaker-user/aep/processed/batch3
upload: processed/batch3/line.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch3/line.parquet
upload: processed/batch3/pole.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch3/pole.parquet


In [5]:
fac = []
for name, typ in [("Hospitals_Medic_Pairwis_Clip", "hospital"), ("Fire_Stations_E_Pairwis_Clip", "fire_station"),
                  ("Ambulance_Servi_Pairwis_Clip", "ambulance")]:
    g = add_latlon(read_shp(name))          # read_shp converts Web Mercator -> lat/lon
    fac.append(pd.DataFrame({"name": g["NAME"], "type": typ, "town": g["CITY"].str.title(),
                             "lat": g["lat"], "lon": g["lon"]}))
facility = pd.concat(fac, ignore_index=True)
facility["is_general_er"] = facility.type.eq("hospital") & ~facility.name.isin(
    ["Catawba Hospital", "Salem Veterans Affairs Medical Center"])
fz = gpd.sjoin_nearest(points(facility), tz, how="left")
facility["zone_id"] = fz[~fz.index.duplicated()]["xf_zone"].values
check(facility, "facility")
print(facility.type.value_counts())

# INTERNAL ONLY: priority per zone = Life Support count + serves a critical facility
zone = pd.read_parquet(f"{OUT}/batch1/zone.parquet")
ls_per_xf = pd.read_parquet(f"{LOCAL}/batch1/life_support_per_transformer.parquet")
ls = ls_per_xf.merge(transformer[["transformer_loc", "zone_id"]], left_on="aep_xfmr_l", right_on="transformer_loc")
device_priority = (zone[["zone_id"]].merge(ls.groupby("zone_id")["life_support_count"].sum().reset_index(),
                                           on="zone_id", how="left").fillna({"life_support_count": 0}))
device_priority["serves_critical_facility"] = device_priority.zone_id.isin(set(facility.zone_id.dropna()))
print("zones with life support customers:", (device_priority.life_support_count > 0).sum(),
      "| zones serving a critical facility:", device_priority.serves_critical_facility.sum())
facility

--- facility: 52 rows
  outside study area: 0
  PRIVACY OK
  null %: {}
type
fire_station    37
ambulance       10
hospital         5
Name: count, dtype: int64
zones with life support customers: 112 | zones serving a critical facility: 49


,name,type,town,lat,lon,is_general_er,zone_id
0,Catawba Hospital,hospital,Catawba,37.391554,-80.110167,False,37800130000121
1,LewisGale Medical Center,hospital,Salem,37.263346,-80.031677,True,37800276A21269
2,Salem Veterans Affairs Medical Center,hospital,Salem,37.274348,-80.018608,False,37800252D10635
3,Carilion Roanoke Memorial Hospital,hospital,Roanoke,37.251217,-79.942274,True,37800278B46997
4,Carilion Roanoke Community Hospital,hospital,Roanoke,37.265672,-79.939458,True,37800278A41978
5,Roanoke Fire - EMS Station 11,fire_station,Roanoke,37.250625,-79.916599,False,37800302C34007
6,Stewartsville - Chamblissburg Volunteer Fire D...,fire_station,Vinton,37.271576,-79.793263,False,37800281C36078
7,Roanoke Fire-EMS Station 2,fire_station,Roanoke,37.289334,-79.937287,False,BREAKER_CIR83
8,Roanoke Fire-EMS Station 14,fire_station,Roanoke,37.294658,-79.900512,False,37800255A10174
9,Vinton Volunteer Fire Department,fire_station,Vinton,37.279587,-79.899081,False,37800255B33039


In [6]:
labor_unit = pd.DataFrame([
    ("pole", "small 25-35ft", 1.942, 0.764), ("pole", "medium 40-55ft", 2.503, 0.927), ("pole", "large 60ft+", 3.368, 1.153),
    ("transformer", "25kVA", 1.188, 0.806), ("transformer", "50kVA", 1.245, 0.891), ("transformer", "100kVA", 1.280, 0.944),
    ("crossarm", None, 0.374, 0.212), ("bracket", None, 0.617, 0.254), ("anchor", None, 0.397, 0.119),
    ("equipment_lead", None, 0.650, 0.206), ("grounding", None, 0.532, 0.153), ("guying", None, 0.367, 0.162),
    ("dead_end", None, 0.108, 0.064), ("insulator", None, 0.049, 0.039), ("pin", None, 0.066, 0.042),
    ("recloser", None, 0.573, 0.525), ("secondary_attachment", None, 0.233, 0.127), ("service", None, 1.693, 0.480),
    ("switch", None, 1.758, 1.010), ("fuse_cutout", None, 0.256, 0.128), ("connector", None, 0.215, 0.045),
], columns=["part", "size_band", "install_h", "removal_h"])

labor_adder = pd.DataFrame([
    ("access_constraint", 0.5), ("confined_property_backyard", 0.5), ("remote_site_offroad", 1.312),
    ("specialized_handling", 0.5), ("site_complexity", 0.5), ("indirect_labor_multiplier_ILM", 4.5),
], columns=["adder", "hours"])

repair_bundle = pd.DataFrame([
    ("Pole", "pole(medium)+crossarm+3 insulator+3 pin+guying+grounding", 5.8),
    ("Transformer", "transformer(25kVA)+fuse_cutout+equipment_lead+grounding", 3.9),
    ("Switch", "switch", 2.8),
    ("Recloser Bank", "recloser+equipment_lead", 2.0),
    ("secondary", "service+secondary_attachment", 2.5),
    ("Fuse Bank", "fuse_cutout", 0.4),
    ("primary", "2 dead_end+2 connector+insulator+pin (ASSUMPTION: no conductor hours in workbook)", 1.1),
], columns=["asset_type", "parts", "direct_hours"])

for df, n in [(facility, "facility"), (labor_unit, "labor_unit"), (labor_adder, "labor_adder"), (repair_bundle, "repair_bundle")]:
    save(df, n, "batch4")
save(device_priority, "device_priority", "batch4", local_only=True)     # stays on this machine
!aws s3 sync ~/aep/processed/batch4 s3://sagemaker-aep-outage-indiansassemble/processed/batch4
!aws s3 ls s3://sagemaker-aep-outage-indiansassemble/processed/ --recursive | wc -l

saved facility: 52 rows -> /home/sagemaker-user/aep/processed/batch4
saved labor_unit: 21 rows -> /home/sagemaker-user/aep/processed/batch4
saved labor_adder: 6 rows -> /home/sagemaker-user/aep/processed/batch4
saved repair_bundle: 7 rows -> /home/sagemaker-user/aep/processed/batch4
saved device_priority: 6,664 rows -> /home/sagemaker-user/aep/local_only/batch4
upload: processed/batch4/labor_adder.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch4/labor_adder.parquet
upload: processed/batch4/labor_unit.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch4/labor_unit.parquet
upload: processed/batch4/repair_bundle.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch4/repair_bundle.parquet
upload: processed/batch4/facility.parquet to s3://sagemaker-aep-outage-indiansassemble/processed/batch4/facility.parquet
15
